### **API Calls and Response structured in Class and sub-class**

In [0]:
import requests
import urllib3
import pandas as pd
import logging
import os
import time
import json
from pyspark.sql import Row
from pyspark.sql.types import StructType, StructField, StringType, TimestampType
from datetime import datetime



class SAP_BO_Connection:
    def __init__(self,bo_url, user, password):
        self.user=user
        self.password=password
        self.bo_url=bo_url
        self.logon_status=False
        self.logonToken=self.logon_request()
        self.headers = {
                        "Content-Type": "application/json",
                        "Accept": "application/json",
                        "User-Agent": "Mozilla/5.0",
                        "X-SAP-LogonToken": self.logonToken
                        }
    
    def logon_request(self):
        logon_url = f"{self.bo_url}/logon/long" 
        auth_type = "secLDAP"
        logon_payload = {
        "userName": self.user,
        "password": self.password,
        "auth": auth_type
        }
        headers = {"Content-Type": "application/json"}
        # logging.info(f"API URL Used for Logon: {logon_url}")
        try:
            response = requests.post(logon_url, json=logon_payload, headers=headers, verify=False)
            # API_CALLS["Total Calls"] += 1
            logon_token = response.headers.get("X-SAP-LogonToken", None)
            print("Logon Token:", logon_token)
            self.logon_status = True if logon_token else False
        except Exception as e:
            print("Failed to logon SAP BO Server:",logon_url," with error: ", e, "exiting script.")
            log_status = False
        return logon_token
   
    def logoff_request(self):
        if self.logon_status is True: 
            logoff_url = f"{self.bo_url}/logoff"
            logoff_headers = {"X-SAP-LogonToken": self.logonToken}
            requests.post(logoff_url, headers=logoff_headers, verify=False)
            self.logon_status = False
            print("Logged off from BO REST API.")
            return True
        else:
            print("Not logged on to BO REST API.")
            return False
    
    
 # Retrieve Document level MetaData
    def get_document(self,document_id):
        if self.logon_status is True:
            document_url = f"{self.bo_url}raylight/v1/documents/{document_id}"
            try:
                document_response = requests.get(document_url, headers=self.headers, timeout=(3,100), verify=False)
                document_data = document_response.json()
                return document_data
            except Exception as e:
                print("Failed to retrieve data:", document_url, "with error:", e, "exiting call.")
                return None
        else:
            print("Not logged on to BO REST API.")
            return None

 # Retrieve Document schedules
    def get_schedules(self, document_id):
        if self.logon_status is True:
            schedules_url = f"{self.bo_url}raylight/v1/documents/{document_id}/schedules"
            try:
                Schedules_response = requests.get(schedules_url, headers=self.headers, timeout=(3,100), verify=False)
                schedules_data = Schedules_response.json()
                return schedules_data
            except Exception as e:
                print("Failed to retrieve data:", schedules_url, "with error:", e, "exiting call.")
                return None
        else:
            print("Not logged on to BO REST API.")
            return None
    # /raylight/v1/documents/21866824
    # Get last schedule instance information
    def get_schedule_details(self, document_id, schdule_id):
        if self.logon_status is True:
            instance_url = f"{self.bo_url}raylight/v1/documents/{document_id}/schedules/{schdule_id}"
            try:
                Instance_response = requests.get(instance_url, headers=self.headers, timeout=(3,100), verify=False)
                schedules_data = Instance_response.json()
                return schedules_data
            except Exception as e:
                print("Failed to retrieve data:", instance_url, "with error:", e, "exiting call.")
                return None
        else:
            print("Not logged on to BO REST API.")
    # /raylight/v1/documents/21866824/schedules/24617393

class webi_Document: #WEBI Document Object to hold the structured details for the MetaData extraction
    def __init__(self, document_id):
        self.document_id = document_id
        self.document_cuid: str = None
        self.document_name: str = None
        self.folder_id: str = None
        self.full_path: str = None
        self.updated: str = None
        self.scheduled: str = None
        self.created: str = None
        self.lastAuthor: str = None
        self.data_providers: list = [] # Container for data providers details
        self.schedules: list = [] # container for schedules
        self.last_instance: ScheduleInstance | None = None # object for the last instance details 
        
    def set_doc_metadata(self, document_data):
        self.document_cuid = document_data["cuid"]
        self.document_name = document_data["name"]
        self.folder_id = document_data["folderId"]
        self.full_path = document_data["path"]
        self.updated = document_data["updated"]
        self.scheduled = document_data["scheduled"]
        self.created = document_data["createdBy"]
        self.lastAuthor = document_data["lastAuthor"]
        
    def set_schedule_instances(self, schedule_instances):
        self.schedules = schedule_instances

    def get_latest_schedule_id(self):
        """
        schedules: list of schedule dicts
        returns: id of the schedule with the latest updated timestamp
        """
        if not self.schedules:
            return None

        latest_item = max(
            self.schedules,
            key=lambda s: datetime.fromisoformat(s["updated"].replace("Z", "+00:00"))
        )

        return latest_item["id"]
    
    def set_latest_schedule_instance(self, last_instance):
        """
        last_instance: details of the latest schedule
        returns: None
        """
        self.last_instance = ScheduleInstance(last_instance)

class ScheduleInstance:
    def __init__(self, schedule_instance_data):
        self.id = schedule_instance_data.get("id")
        self.name = schedule_instance_data.get("name")
        self.updated = schedule_instance_data.get("updated")
        self.format = schedule_instance_data.get("format", {}).get("@type")
        self.status = schedule_instance_data.get("status", {}).get("$")
        self.parameters=schedule_instance_data.get("parameters", {}) # please use structure to save this into database
        self.repetition_type: str = None 
        self.repetition: dict = {} # please use structure to save this into database, it should be dictionary
        self.destination_type: str = None 
        self.destination: dict = {} # please use structure to save this into database, it should be dictionary
        self.get_repetition(schedule_instance_data)
        self.get_destination(schedule_instance_data)

    def get_repetition(self, schedule_instance_data):
        
        SCHEDULE_KEYS = {
            "once",
            "hourly",
            "daily",
            "weekly",
            "monthly",
            "calendar"
        }
        completed=False
        for key in SCHEDULE_KEYS:
            if key in schedule_instance_data:
                self.repetition_type = key
                self.repetition = schedule_instance_data[key]
                completed=True
                return None

        if not completed:
            print("No repetition type found in the schedule instance data.")
        return None
    
    def get_destination(self, schedule_instance_data):

        DESTINATION_KEYS = {
            "mail",
            "inbox",
            "file",
            "ftp"
        }

        completed = False
        destination_block = schedule_instance_data.get("destination", {})

        for key in DESTINATION_KEYS:
            if key in destination_block:
                self.destination_type = key
                self.destination = destination_block[key]
                completed = True
                return None

        if not completed:
            print("No destination type found in the schedule instance data.")

        return None
    
    def __repr__(self):
        return (
            f"ScheduleInstance(,\n"
            f"id={self.id}, \n"
            f"name={self.name}, \n"
            f"updated={self.updated}, \n"
            f"format is {self.format}, \n"
            f"with status {self.status}, \n"
            f"repetition_type={self.repetition_type}, \n"
            f"destination_type={self.destination_type}"
            f")"
        )


def process_document(webi_id:str,bo_connection:SAP_BO_Connection):
    Document_data=bo_connection.get_document(webi_id)
    WEBI_doc=webi_Document(webi_id)
    # print(Document_data)
    WEBI_doc.set_doc_metadata(Document_data["document"])
    print(WEBI_doc.document_name)
    if(WEBI_doc.scheduled=="true"):
        print("Document is scheduled")
        schedule_package = bo_connection.get_schedules(WEBI_doc.document_id)
        WEBI_doc.set_schedule_instances(schedule_package['schedules']['schedule'])
        # print(WEBI_doc.schedules)
        print("\n")
        last_instance_data=bo_connection.get_schedule_details(WEBI_doc.document_id, WEBI_doc.get_latest_schedule_id())
        # print(last_instance_data)
        WEBI_doc.set_latest_schedule_instance(last_instance_data['schedule'])
        print(WEBI_doc.last_instance)
    else:
        print("Document is not scheduled")
    return WEBI_doc

def persist_webi_document(
    spark,
    doc: webi_Document,
    table_name: str = "dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.webi_schedule_details"
):
    """
    Inserts or updates WebI document + schedule instance data into Databricks Delta table.
    """
    if not doc.last_instance:
        raise ValueError("No last schedule instance available for persistence")
    print("\n")
    print("persisting document and schedule instance")
    inst = doc.last_instance
        # -----------------------------
    # Build row payload
    # -----------------------------
    row_data = {
        "document_id": str(doc.document_id),
        "document_cuid": doc.document_cuid,
        "document_name": doc.document_name,
        "folder_id": str(doc.folder_id) if doc.folder_id is not None else None,
        "full_path": doc.full_path,

        "document_updated_ts": doc.updated,
        "document_scheduled": doc.scheduled,
        "document_created_by": doc.created,
        "document_last_author": doc.lastAuthor,

        "schedule_id": str(inst.id) if inst.id is not None else None,
        "schedule_name": inst.name,

        "schedule_updated_ts": inst.updated,
        "schedule_format": inst.format,
        "schedule_status": inst.status,

        "repetition_type": inst.repetition_type,
        "repetition": json.dumps(inst.repetition),

        "destination_type": inst.destination_type,
        "destination": json.dumps(inst.destination),

        "parameters": json.dumps(inst.parameters),
        "ingestion_ts": datetime.utcnow()
    }

    schema = StructType([
        StructField("document_id", StringType()),
        StructField("document_cuid", StringType()),
        StructField("document_name", StringType()),
        StructField("folder_id", StringType()),
        StructField("full_path", StringType()),
        StructField("document_updated_ts", StringType()),
        StructField("document_scheduled", StringType()),
        StructField("document_created_by", StringType()),
        StructField("document_last_author", StringType()),
        StructField("schedule_id", StringType()),
        StructField("schedule_name", StringType()),
        StructField("schedule_updated_ts", StringType()),
        StructField("schedule_format", StringType()),
        StructField("schedule_status", StringType()),
        StructField("repetition_type", StringType()),
        StructField("repetition", StringType()),
        StructField("destination_type", StringType()),
        StructField("destination", StringType()),
        StructField("parameters", StringType()),
        StructField("ingestion_ts", TimestampType()),
    ])
    df = spark.createDataFrame([row_data], schema=schema)
    df.createOrReplaceTempView("webi_stage")

    # -----------------------------
    # MERGE (UPSERT)
    # -----------------------------
    spark.sql(f"""
        MERGE INTO {table_name} t
        USING webi_stage s
        ON  t.document_id = s.document_id
        AND t.schedule_id = s.schedule_id
        WHEN MATCHED THEN UPDATE SET *
        WHEN NOT MATCHED THEN INSERT *
    """)
    update_tracker(spark, report_id=int(doc.document_id), schedule_id=inst.id)
    return None

def update_tracker(
    spark,
    report_id: int,
    schedule_id: str,
    result_text: str = "Persisted; ",
    tracker_table: str = "dataplatform01_central_dev_catalog_01.custom_sap_bo.webi_schedule_scan_tracker"
):
    scanned_date = datetime.utcnow().isoformat()
    print(f"Updating tracker for report_id={report_id}, schedule_id={schedule_id}")
    spark.sql(f"""
        UPDATE {tracker_table}
        SET
            Scanned = 'Y',
            Scanned_Date = '{scanned_date}',
            CMS_scheduled = true,
            Result = '{result_text} | Schedule_ID={schedule_id}'
        WHERE Report_ID = {report_id}
    """)
    return None

def main():
    urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)
    Todyay_Date = datetime.now().strftime("%Y-%m-%d")

    user="ESBWIL1"
    password="Built2@escape"
    logging.info(f"Script started.")
    bo_url = "https://wavgbcdfp1088.atradiusnet.com:8443/biprws/"
    bo_connection = SAP_BO_Connection(bo_url,user,password)
    webi=process_document("21842960",bo_connection)
    print("\n retrieved content from REST API, continue persist into DB")
    persist_webi_document(spark=spark,doc=webi)
    print("\nLogged off: ",bo_connection.logoff_request())


if __name__ == "__main__":
    main()

### _Retrieve SQL to form a collection to run through_

watch out for record tracking, and when restart process

In [0]:
import requests
import urllib3
import pandas as pd
import logging
import os
import time
import json
from pyspark.sql import Row
from pyspark.sql.types import StructType, StructField, StringType, TimestampType, LongType
from datetime import datetime



class SAP_BO_Connection:
    def __init__(self,bo_url, user, password):
        self.user=user
        self.password=password
        self.bo_url=bo_url
        self.logon_status=False
        self.logonToken=self.logon_request()
        self.headers = {
                        "Content-Type": "application/json",
                        "Accept": "application/json",
                        "User-Agent": "Mozilla/5.0",
                        "X-SAP-LogonToken": self.logonToken
                        }
    
    def logon_request(self):
        logon_url = f"{self.bo_url}/logon/long" 
        auth_type = "secLDAP"
        logon_payload = {
        "userName": self.user,
        "password": self.password,
        "auth": auth_type
        }
        headers = {"Content-Type": "application/json"}
        # logging.info(f"API URL Used for Logon: {logon_url}")
        try:
            response = requests.post(logon_url, json=logon_payload, headers=headers, verify=False)
            # API_CALLS["Total Calls"] += 1
            logon_token = response.headers.get("X-SAP-LogonToken", None)
            print("Logon Token:", logon_token)
            self.logon_status = True if logon_token else False
        except Exception as e:
            print("Failed to logon SAP BO Server:",logon_url," with error: ", e, "exiting script.")
            self.logon_status = False
        return logon_token
   
    def logoff_request(self):
        if self.logon_status is True: 
            logoff_url = f"{self.bo_url}/logoff"
            logoff_headers = {"X-SAP-LogonToken": self.logonToken}
            response = requests.post(logoff_url, headers=logoff_headers, verify=False)
            print("Logoff response code:", response.status_code)
            self.logon_status = False
            return True
        else:
            print("Not logged on to BO REST API.")
            return False
    
    
 # Retrieve Document level MetaData
    def get_document(self,document_id):
        if self.logon_status is True:
            document_url = f"{self.bo_url}raylight/v1/documents/{document_id}"
            try:
                document_response = requests.get(document_url, headers=self.headers, timeout=(3,100), verify=False)
                document_data = document_response.json()
                return document_data
            except Exception as e:
                print("Failed to retrieve data:", document_url, "with error:", e, "exiting call.")
                return None
        else:
            print("Not logged on to BO REST API.")
            return None

 # Retrieve Document schedules
    def get_schedules(self, document_id):
        if self.logon_status is True:
            schedules_url = f"{self.bo_url}raylight/v1/documents/{document_id}/schedules"
            try:
                Schedules_response = requests.get(schedules_url, headers=self.headers, timeout=(3,100), verify=False)
                schedules_data = Schedules_response.json()
                return schedules_data
            except Exception as e:
                print("Failed to retrieve data:", schedules_url, "with error:", e, "exiting call.")
                return None
        else:
            print("Not logged on to BO REST API.")
            return None
    # /raylight/v1/documents/21866824
    # Get last schedule instance information
    def get_schedule_details(self, document_id, schdule_id):
        if self.logon_status is True:
            instance_url = f"{self.bo_url}raylight/v1/documents/{document_id}/schedules/{schdule_id}"
            try:
                Instance_response = requests.get(instance_url, headers=self.headers, timeout=(3,100), verify=False)
                schedules_data = Instance_response.json()
                return schedules_data
            except Exception as e:
                print("Failed to retrieve data:", instance_url, "with error:", e, "exiting call.")
                return None
        else:
            print("Not logged on to BO REST API.")
    # /raylight/v1/documents/21866824/schedules/24617393

class webi_Document: #WEBI Document Object to hold the structured details for the MetaData extraction
    def __init__(self, document_id):
        self.document_id = document_id
        self.document_cuid: str = None
        self.document_name: str = None
        self.folder_id: str = None
        self.full_path: str = None
        self.updated: str = None
        self.scheduled: str = None
        self.created: str = None
        self.lastAuthor: str = None
        self.data_providers: list = [] # Container for data providers details
        self.schedules: list = [] # container for schedules
        self.last_instance: ScheduleInstance | None = None # object for the last instance details 
        
    def set_doc_metadata(self, document_data):
        self.document_cuid = document_data.get("cuid")
        self.document_name = document_data.get("name")
        self.folder_id = document_data.get("folderId")
        self.full_path = document_data.get("path")
        self.updated = document_data.get("updated")
        self.scheduled = document_data.get("scheduled")
        self.created = document_data.get("createdBy")
        self.lastAuthor = document_data.get("lastAuthor")
        
    def set_schedule_instances(self, schedule_instances):
        self.schedules = schedule_instances

    def get_latest_schedule_id(self):
        """
        schedules: list of schedule dicts
        returns: id of the schedule with the latest updated timestamp
        """
        if not self.schedules:
            return None

        latest_item = max(
            self.schedules,
            key=lambda s: datetime.fromisoformat(s["updated"].replace("Z", "+00:00"))
        )

        return latest_item["id"]
    
    def set_latest_schedule_instance(self, last_instance):
        """
        last_instance: details of the latest schedule
        returns: None
        """
        self.last_instance = ScheduleInstance(last_instance)

class ScheduleInstance:
    def __init__(self, schedule_instance_data):
        self.id = schedule_instance_data.get("id")
        self.name = schedule_instance_data.get("name")
        self.updated = schedule_instance_data.get("updated")
        self.format = schedule_instance_data.get("format", {}).get("@type")
        self.status = schedule_instance_data.get("status", {}).get("$")
        self.parameters=schedule_instance_data.get("parameters", {}) # please use structure to save this into database
        self.repetition_type: str = None 
        self.repetition: dict = {} # please use structure to save this into database, it should be dictionary
        self.destination_type: str = None 
        self.destination: dict = {} # please use structure to save this into database, it should be dictionary
        self.get_repetition(schedule_instance_data)
        self.get_destination(schedule_instance_data)
        self.json = json.dumps(schedule_instance_data)

    def get_repetition(self, schedule_instance_data):
        
        SCHEDULE_KEYS = {
            "once",
            "hourly",
            "daily",
            "weekly",
            "monthly",
            "calendar"
        }
        completed=False
        for key in SCHEDULE_KEYS:
            if key in schedule_instance_data:
                self.repetition_type = key
                self.repetition = schedule_instance_data[key]
                completed=True
                return None

        if not completed:
            print("No repetition type found in the schedule instance data.")
        return None
    
    def get_destination(self, schedule_instance_data):

        DESTINATION_KEYS = {
            "mail",
            "inbox",
            "file",
            "ftp",
            "filesystem"
        }

        completed = False
        destination_block = schedule_instance_data.get("destination", {})

        for key in DESTINATION_KEYS:
            if key in destination_block:
                self.destination_type = key
        
        self.destination = destination_block
        return None
    
    def __repr__(self):
        return (
            f"ScheduleInstance(,\n"
            f"id={self.id}, \n"
            f"name={self.name}, \n"
            f"updated={self.updated}, \n"
            f"format is {self.format}, \n"
            f"with status {self.status}, \n"
            f"repetition_type={self.repetition_type}, \n"
            f"destination_type={self.destination_type}"
            f")"
        )


def process_document(webi_id:str,bo_connection:SAP_BO_Connection):
    Document_data=bo_connection.get_document(webi_id)
    WEBI_doc=webi_Document(webi_id)
    # print(Document_data)
    if Document_data is None or "document" not in Document_data:
        print(f"Warning: No valid response for document {webi_id}. API returned: {Document_data}")
        return WEBI_doc
    WEBI_doc.set_doc_metadata(Document_data["document"])
    # print(WEBI_doc.document_name)
    if(WEBI_doc.scheduled=="true"):
        # print("Document is scheduled")
        schedule_package = bo_connection.get_schedules(WEBI_doc.document_id)
        if schedule_package is None or "schedules" not in schedule_package or "schedule" not in schedule_package.get("schedules", {}):
            print(f"Warning: No valid schedules response for document {webi_id}. API returned: {schedule_package}")
            return WEBI_doc
        WEBI_doc.set_schedule_instances(schedule_package.get("schedules", {}).get("schedule", {}))
        last_instance_data=bo_connection.get_schedule_details(WEBI_doc.document_id, WEBI_doc.get_latest_schedule_id())
        if last_instance_data is None or "schedule" not in last_instance_data:
            print(f"Warning: No valid schedule details response for document {webi_id}. API returned: {last_instance_data}")
            return WEBI_doc
        WEBI_doc.set_latest_schedule_instance(last_instance_data['schedule'])
    else:
        print("Document is not scheduled")
    return WEBI_doc

def persist_webi_documents(
    spark,
    docs: list,
    table_name: str = "dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.webi_schedule_details"):
    """
    Batch inserts or updates WebI document + schedule instance data into Databricks Delta table.
    """
    rows = []
    for doc in docs:
        if not doc.last_instance:
            print(f"Skipping document {doc.document_id} - no schedule instance")
            continue
        inst = doc.last_instance
        rows.append({
            "document_id": str(doc.document_id),
            "document_cuid": doc.document_cuid,
            "document_name": doc.document_name,
            "folder_id": str(doc.folder_id) if doc.folder_id is not None else None,
            "full_path": doc.full_path,
            "document_updated_ts": doc.updated,
            "document_scheduled": doc.scheduled,
            "document_created_by": doc.created,
            "document_last_author": doc.lastAuthor,
            "schedule_id": str(inst.id) if inst.id is not None else None,
            "schedule_name": inst.name,
            "schedule_updated_ts": inst.updated,
            "schedule_format": inst.format,
            "schedule_status": inst.status,
            "repetition_type": inst.repetition_type,
            "repetition": json.dumps(inst.repetition),
            "destination_type": inst.destination_type,
            "destination": json.dumps(inst.destination),
            "parameters": json.dumps(inst.parameters),
            "instance_json": inst.json,
            "ingestion_ts": datetime.utcnow()
        })

    if not rows:
        print("No documents with schedule instances to persist")
        return None

    schema = StructType([
        StructField("document_id", StringType()),
        StructField("document_cuid", StringType()),
        StructField("document_name", StringType()),
        StructField("folder_id", StringType()),
        StructField("full_path", StringType()),
        StructField("document_updated_ts", StringType()),
        StructField("document_scheduled", StringType()),
        StructField("document_created_by", StringType()),
        StructField("document_last_author", StringType()),
        StructField("schedule_id", StringType()),
        StructField("schedule_name", StringType()),
        StructField("schedule_updated_ts", StringType()),
        StructField("schedule_format", StringType()),
        StructField("schedule_status", StringType()),
        StructField("repetition_type", StringType()),
        StructField("repetition", StringType()),
        StructField("destination_type", StringType()),
        StructField("destination", StringType()),
        StructField("parameters", StringType()),
        StructField("instance_json", StringType()),
        StructField("ingestion_ts", TimestampType()),
    ])
    df = spark.createDataFrame(rows, schema=schema)
    df.createOrReplaceTempView("webi_stage")

    # -----------------------------
    # MERGE (UPSERT)
    # -----------------------------
    spark.sql(f"""
        MERGE INTO {table_name} t
        USING webi_stage s
        ON  t.document_id = s.document_id
        AND t.schedule_id = s.schedule_id
        WHEN MATCHED THEN UPDATE SET *
        WHEN NOT MATCHED THEN INSERT *
    """)
    print(f"Persisted {len(rows)} documents into {table_name}")
    return None

def update_tracker(
    spark,
    tracker_entries: list,
    tracker_table: str = "dataplatform01_central_dev_catalog_01.custom_sap_bo.webi_schedule_scan_tracker"):
    if not tracker_entries:
        return None
    scanned_date = datetime.utcnow().isoformat()
    tracker_rows = []
    for report_id, schedule_id, scanned in tracker_entries:
        tracker_rows.append({
            "Report_ID": report_id,
            "schedule_id_val": str(schedule_id) if schedule_id is not None else "",
            "scanned_date_val": scanned_date,
            "scanned_val": scanned
        })
        # print(f"Updating tracker for report_id={report_id}, schedule_id={schedule_id}, scanned={scanned}")

    tracker_schema = StructType([
        StructField("Report_ID", LongType()),
        StructField("schedule_id_val", StringType()),
        StructField("scanned_date_val", StringType()),
        StructField("scanned_val", StringType()),
    ])
    df = spark.createDataFrame(tracker_rows, schema=tracker_schema)
    df.createOrReplaceTempView("tracker_stage")

    spark.sql(f"""
        MERGE INTO {tracker_table} t
        USING tracker_stage s
        ON t.Report_ID = s.Report_ID
        WHEN MATCHED THEN UPDATE SET
            t.Scanned = s.scanned_val,
            t.Scanned_Date = s.scanned_date_val,
            t.CMS_scheduled = true,
            t.Result = concat('Persisted; | Schedule_ID=', s.schedule_id_val)
    """)
    print(f"Updated tracker for {len(tracker_entries)} reports")
    return None

def main():
    urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)
    user="ESBWIL1"
    password="Built2@escape"
    logging.info(f"Script started.")
    bo_url = "https://wavgbcdfp1088.atradiusnet.com:8443/biprws/"
    bo_connection = SAP_BO_Connection(bo_url,user,password)
    df_ids = spark.sql("""
                       SELECT distinct Report_ID FROM dataplatform01_central_dev_catalog_01.custom_sap_bo.webi_schedule_scan_tracker
                       WHERE Scanned = '' and CMS_scheduled=true
                       ORDER BY Report_ID DESC
                    --    LIMIT 400;
                       """)
    id_iterator = df_ids.toLocalIterator()
    webi_array=[]
    tracker_entries = []

    for row in id_iterator:
        print(row)
        webi=process_document(str(row[0]),bo_connection)
        if webi.document_name is None:
            tracker_entries.append((int(webi.document_id), str(webi.last_instance.id) if webi.last_instance is not None else None, "N"))
            continue
        webi_array.append(webi)
        tracker_entries.append((int(webi.document_id), str(webi.last_instance.id) if webi.last_instance is not None else None, "Y"))
    
    print("\nLogged off status: ",bo_connection.logoff_request())
    print(f"\nCollected {len(webi_array)} documents in webi_array")
    persist_webi_documents(spark=spark, docs=webi_array)
    update_tracker(spark, tracker_entries=tracker_entries)

    return webi_array


if __name__ == "__main__":
    webi_array = main()

